In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load preprocessed fire df for aggregration
fire_df = pd.read_csv('/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Fire_LC_Dataset.csv')
fire = fire_df.copy()
print(fire.columns)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Index(['Fire_ID', 'Acquisition_Date', 'Acquisition_Time', 'Year', 'Month',
       'Day_of_Year', 'Hour', 'Latitude_Fire', 'Longitude_Fire', 'Bright_TI4',
       'Bright_TI5', 'FRP', 'Scan', 'Track', 'Confidence', 'DayNight',
       'Fire_Type', 'LC_Type1', 'LC_Label'],
      dtype='object')


### Set up columns and keys for aggregation

In [ ]:
# Check dates formatted properly
date_check = pd.to_datetime(fire['Acquisition_Date'], errors='coerce')
print("Unreadable Dates:", date_check.isna().sum())
print(date_check.head())

# Check entries for confidence column for mapping
fire['Confidence'].unique()

Unreadable Dates: 0
0   2014-01-01
1   2014-01-01
2   2014-01-01
3   2014-01-01
4   2014-01-01
Name: Acquisition_Date, dtype: datetime64[ns]


array(['n', 'h', 'l'], dtype=object)

In [ ]:
# Setup date column
fire["Acquisition_Date"] = pd.to_datetime(fire["Acquisition_Date"]).dt.date

# Set keys
key_columns = ["Latitude_Fire", "Longitude_Fire", "Acquisition_Date"]

# Map confidence level to ordered numbers
confidence_map = {
    "l": 1,     # low
    "n": 2,     # nominal
    "h": 3      # high
}
fire["Confidence_Ordered"] = fire["Confidence"].map(confidence_map)

### Apply aggregation to find representative records

In [ ]:
# Sorts the best row for each location-day to the top
fire_sorted = fire.sort_values(
    by=key_columns + ["FRP", "Confidence_Ordered"],
    ascending=[True, True, True, False, False]
)

# Checks sorted df and keeps first row for each (lat, long, date) combination
rep = fire_sorted.drop_duplicates(subset=key_columns, keep="first")

# Generate group-level summaries to validate aggregation
summary = (
    fire.groupby(key_columns)             # groups by keys
    .agg(
        detection_count=("FRP", "size"),  # count of how many rows aggregated
        FRP_max=("FRP", "max")            # identifies max FRP in group
    )
    .reset_index()
)

# Combine representative rows + summaries
fire_daily = rep.merge(summary, on=key_columns, how="left")


### Validation checks

In [ ]:
# Compare row counts before and after aggregation
print("Original Rows:", len(fire_df))
print("Aggregated Rows:", len(fire_daily))

# Check for duplicates
print("Duplicates after Aggregation:",
      fire_daily.duplicated(subset=key_columns).sum())

# Check sum of all detection counts equals original detection counts
print("Sum(detection_count):",
      fire_daily["detection_count"].sum())

# Check representative FRP equals max FRP
print("Representative FRP != FRP_max:",
      (fire_daily["FRP"] != fire_daily["FRP_max"]).sum())

# Check how many detections fall into each detection count category
print("Detections/Detection Count Categor:",
      fire_daily["detection_count"].value_counts().sort_index())

fire_daily.head()

Original Rows: 2300617
Aggregated Rows: 2300610
Duplicates after Aggregation: 0
Sum(detection_count): 2300617
Representative FRP != FRP_max: 0
Detections/Detection Count Categor: detection_count
1    2300603
2          7
Name: count, dtype: int64


,Fire_ID,Acquisition_Date,Acquisition_Time,Year,Month,Day_of_Year,Hour,Latitude_Fire,Longitude_Fire,Bright_TI4,...,Scan,Track,Confidence,DayNight,Fire_Type,LC_Type1,LC_Label,Confidence_Ordered,detection_count,FRP_max
0,2255440,2024-08-11,1114,2024,8,224,11,48.30000,-120.57123,308.78,...,0.49,0.65,n,N,0,9,Savannas,2,1,2.70
1,935127,2019-10-09,2152,2019,10,282,21,48.30000,-118.81944,356.92,...,0.57,0.69,l,D,0,8,Woody Savannas,1,1,164.97
2,251428,2015-08-23,1003,2015,8,235,10,48.30001,-118.86491,332.71,...,0.38,0.36,n,N,0,8,Woody Savannas,2,1,2.09
3,1024549,2021-07-18,1114,2021,7,199,11,48.30001,-118.53941,317.18,...,0.58,0.70,n,N,0,8,Woody Savannas,2,1,13.97
4,660545,2018-08-14,1018,2018,8,226,10,48.30001,-116.10197,296.26,...,0.56,0.43,n,N,0,1,Evergreen Needleleaf Forests,2,1,1.35


### Quick fire dataset checks before final merge

In [ ]:
# Check coordinate range
print("Latitude range:",
      fire_daily["Latitude_Fire"].min(),
      fire_daily["Latitude_Fire"].max())

print("Longitude range:",
      fire_daily["Longitude_Fire"].min(),
      fire_daily["Longitude_Fire"].max())

# Check date coverage
fire_daily["Acquisition_Date"].min(), fire_daily["Acquisition_Date"].max()

# Check number of unique fire locations
print("Unique fire locations:",
      fire_daily[["Latitude_Fire", "Longitude_Fire"]].drop_duplicates().shape[0])

# Check rough ranges for FRP values
fire_daily["FRP_max"].describe(percentiles=[0.9, 0.95, 0.99])

Latitude range: 48.3 60.0
Longitude range: -138.23932 -108.00001
Unique fire locations: 2300523


,FRP_max
count,2.300610e+06
mean,1.532790e+01
std,3.988171e+01
min,-2.000000e-01
50%,4.300000e+00
90%,3.341000e+01
95%,6.443000e+01
99%,1.909091e+02
max,4.191100e+03


In [ ]:
# Save final merge-ready fire dataset to Google Drive as CSV
save_path = '/content/drive/MyDrive/MDP_Capstone_Project/Data/Final_Merge_Fire_Dataset.csv'
fire_daily.to_csv(save_path, index=False)